# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


## Bayesian Estimation of a User Ability Parameter from Item Responses — Solution

### Task 1: Visualizing the Mechanics

Plotted $P(Y_i=1\mid\Theta=\theta)=\dfrac{1}{1+e^{-a_i(\theta-b_i)}}$ for $a_i\in\{1.0,\,2.5\}$, with $a_i=1.0$ paired against $b_i\in\{-2,0,2\}$ (see accompanying code).

**Interpretation.** With $a_i$ fixed, changing $b_i$ **translates** the curve rigidly along the $\theta$-axis: the curve always passes through $P=0.5$ exactly at $\theta=b_i$, so a larger $b_i$ pushes the entire curve to the right (the item becomes harder — higher ability is required to reach any given success probability), while the shape and slope of the curve are unaffected. Changing $a_i$ instead controls the **steepness** at that crossover point, not its location.

---

### Task 2: Sequential Likelihood Contribution

The single new response $y_k$ is Bernoulli with success probability $p_k(\theta)$, so its likelihood contribution is

$$
L(y_k \mid \theta) = p_k(\theta)^{y_k}\,\big[1-p_k(\theta)\big]^{1-y_k},
\qquad
p_k(\theta) = \frac{1}{1+e^{-a_k(\theta-b_k)}}.
$$

Assuming conditional independence of responses given $\theta$ (the standard IRT local-independence assumption), the joint likelihood of the entire running history $\mathbf{y}^{(k)}=(y_1,\dots,y_k)$ is the product of the per-item terms:

$$
L\big(\mathbf{y}^{(k)} \mid \theta\big) = \prod_{i=1}^{k} p_i(\theta)^{y_i}\,\big[1-p_i(\theta)\big]^{1-y_i}.
$$

---

### Task 3: Recursive Posterior Update

By Bayes' rule, using $f_{\Theta\mid\mathbf{Y}^{(k-1)}}(\theta\mid\mathbf{y}^{(k-1)})$ as the prior for step $k$ and $L(y_k\mid\theta)$ as the incremental likelihood:

$$
f_{\Theta\mid\mathbf{Y}^{(k)}}\big(\theta \mid \mathbf{y}^{(k)}\big)
\;\propto\;
f_{\Theta\mid\mathbf{Y}^{(k-1)}}\big(\theta \mid \mathbf{y}^{(k-1)}\big)\cdot L(y_k\mid\theta).
$$

Explicitly, with the normalizing constant made explicit:

$$
f_{\Theta\mid\mathbf{Y}^{(k)}}\big(\theta \mid \mathbf{y}^{(k)}\big)
=
\frac{f_{\Theta\mid\mathbf{Y}^{(k-1)}}(\theta)\,p_k(\theta)^{y_k}\big[1-p_k(\theta)\big]^{1-y_k}}
{\displaystyle\int_{-\infty}^{\infty} f_{\Theta\mid\mathbf{Y}^{(k-1)}}(\theta')\,p_k(\theta')^{y_k}\big[1-p_k(\theta')\big]^{1-y_k}\,d\theta'}.
$$

This is a Bayesian filter: each new observation reweights the previous posterior rather than requiring a full re-derivation from scratch.

---

### Task 4: Dynamic Shifting

Suppose $y_k=1$ (correct) on an item with a large $b_k$ (a hard item). The likelihood factor $p_k(\theta)$ is itself a logistic curve centered at $\theta=b_k$: it is close to $0$ for $\theta \ll b_k$ and only becomes appreciable once $\theta \gtrsim b_k$.

Multiplying the running posterior $f_{\Theta\mid\mathbf{Y}^{(k-1)}}(\theta)$ pointwise by this increasing factor **suppresses posterior mass at low $\theta$** (where the likelihood is near zero) while **preserving or amplifying mass near and above $b_k$**. The net effect, after renormalizing, is that the peak (mode) of the posterior moves to the **right**, toward higher ability — and the size of that shift grows with how large $b_k$ is, since a correct answer on a very hard item is strong evidence of high ability.

---

### Task 5: Tracking Certainty and Sharpness

The discrimination parameter $a_k$ controls the *slope* of $p_k(\theta)$ around $\theta=b_k$, i.e., how much information a single response carries.

- **Very large $a_k$:** $p_k(\theta)$ approaches a step function (near $0$ below $b_k$, near $1$ above it). The likelihood factor then acts almost like a hard cutoff, sharply cutting away posterior mass on the "wrong" side of $b_k$. This produces a large drop in posterior variance — a decisive, high-information update.
- **Very small $a_k$:** $p_k(\theta)$ is nearly flat (close to a constant near $0.5$) across the plausible range of $\theta$. The likelihood factor is then nearly uninformative, and multiplying the posterior by it barely changes its shape or variance — a weak, low-information update.

In short, $a_k$ governs the *sharpness* of each incremental update, while $b_k$ (Task 4) governs *where* the update shifts probability mass.

---

### Task 6: Numerical Implementation of a Running Grid

1. **Discretize.** Fix a grid $\theta_1 < \theta_2 < \cdots < \theta_M$ over a plausible range (e.g., $[-4,4]$ with a few hundred points).
2. **Initialize.** Evaluate the prior on the grid: $f^{(0)}[\theta_m] = \dfrac{1}{\sqrt{2\pi}}e^{-\theta_m^2/2}$.
3. **Per-item update.** After observing $y_k$ for an item with known $(a_k, b_k)$:
   - Compute $p_k(\theta_m)$ for every grid point.
   - Form the likelihood vector $L_k[\theta_m] = p_k(\theta_m)^{y_k}\big[1-p_k(\theta_m)\big]^{1-y_k}$.
   - Multiply elementwise: $g[\theta_m] = f^{(k-1)}[\theta_m]\cdot L_k[\theta_m]$.
4. **Sequential normalization.** Numerically integrate $g$ over the grid (trapezoidal rule) to get the normalizing constant $Z_k = \int g(\theta)\,d\theta$, then set $f^{(k)}[\theta_m] = g[\theta_m]/Z_k$.
5. **Iterate.** $f^{(k)}$ becomes the prior for step $k+1$; repeat steps 3–4 for each new item.
6. **Extract estimators at any step:** posterior mean $\hat\theta_{\text{Bayes}}^{(k)} = \int \theta\, f^{(k)}(\theta)\,d\theta$ (trapezoidal sum $\sum_m \theta_m f^{(k)}[\theta_m]\,\Delta\theta$), and MAP $\hat\theta_{\text{MAP}}^{(k)} = \arg\max_{\theta_m} f^{(k)}[\theta_m]$.

---

### Task 7: Evaluating Convergence over the Timeline

See the accompanying Python code for the full simulation ($\theta_{\text{true}}=0.75$, $n=20$ items, $b_k\sim\mathcal N(0,1)$, $a_k\sim U(0.5,2.0)$).

**Analysis.** Both $\hat\theta_{\text{Bayes}}^{(k)}$ and $\hat\theta_{\text{MAP}}^{(k)}$ start at (or near) $0$ — the prior mean/mode — since little evidence has accumulated. As $k$ increases they drift toward $\theta_{\text{true}}=0.75$, but not monotonically: an unlucky miss on an easy item, or a lucky guess on a hard one, can transiently pull an estimate away from the truth. Because each per-item likelihood narrows the posterior (Task 5), the *rate* at which new evidence can move the estimate slows down as $k$ grows — the posterior variance shrinks, so later responses have progressively less leverage on the running mean/mode than early ones did. This reflects the platform's growing **confidence**: with more responses collected, its estimate of the user's ability stabilizes and becomes increasingly resistant to being shifted by any single subsequent item.


In [ ]:
"""
Bayesian Estimation of a User Ability Parameter from Item Responses
Python code (Tasks 1 and 7) — run cells in Colab, or execute as a script.
Requires: pip install plotly numpy
"""

import numpy as np
import plotly.graph_objects as go

# =====================================================================
# TASK 1 — Visualizing the Mechanics: P(Y_i = 1 | Theta = theta) vs theta
# =====================================================================

def p_item(theta, a, b):
    """2PL item response probability."""
    return 1.0 / (1.0 + np.exp(-a * (theta - b)))

theta = np.linspace(-6, 6, 400)

a_low, a_high = 1.0, 2.5          # two distinct discrimination values
b_values = [-2.0, 0.0, 2.0]       # three difficulties, paired with a_low

fig1 = go.Figure()

for b in b_values:
    fig1.add_trace(go.Scatter(
        x=theta, y=p_item(theta, a_low, b),
        mode="lines",
        name=f"a={a_low}, b={b:+.0f}"
    ))

# second discrimination value, held at the middle difficulty for comparison
fig1.add_trace(go.Scatter(
    x=theta, y=p_item(theta, a_high, 0.0),
    mode="lines",
    name=f"a={a_high}, b=0 (steeper)",
    line=dict(dash="dash", width=3)
))

fig1.update_layout(
    title="Item Characteristic Curves: P(Y_i=1 | Theta=theta) = 1/(1+e^(-a(theta-b)))",
    xaxis_title="theta (ability)",
    yaxis_title="P(Y_i = 1 | theta)",
    template="plotly_white",
    legend_title="item parameters",
    width=850, height=550
)
fig1.add_hline(y=0.5, line_dash="dot", line_color="gray", opacity=0.5)
fig1.show()  # in Colab this renders inline; also save to file if desired
# fig1.write_html("task1_icc_plot.html")


# =====================================================================
# TASK 7 — Evaluating Convergence over the Timeline (grid-based Bayes update)
# =====================================================================

rng = np.random.default_rng(42)

# ---- grid setup -----------------------------------------------------
theta_grid = np.linspace(-4, 4, 801)

def normal_pdf(x, mu=0.0, sigma=1.0):
    return np.exp(-0.5 * ((x - mu) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))

posterior = normal_pdf(theta_grid)                       # f_Theta^(0)(theta)
posterior /= np.trapezoid(posterior, theta_grid)         # normalize

# ---- simulation setup -------------------------------------------------
theta_true = 0.75
n_items = 20

bayes_history = []
map_history = []

for k in range(1, n_items + 1):
    # random item parameters
    a_k = rng.uniform(0.5, 2.0)
    b_k = rng.normal(0.0, 1.0)

    # simulate response at the TRUE ability
    p_true = p_item(theta_true, a_k, b_k)
    u = rng.uniform(0.0, 1.0)
    y_k = 1 if u < p_true else 0

    # likelihood of y_k over the whole theta grid
    p_grid = p_item(theta_grid, a_k, b_k)
    likelihood = p_grid**y_k * (1 - p_grid)**(1 - y_k)

    # Bayesian update: posterior_k proportional to posterior_{k-1} * likelihood
    unnormalized = posterior * likelihood
    posterior = unnormalized / np.trapezoid(unnormalized, theta_grid)

    # running estimators
    theta_bayes = np.trapezoid(theta_grid * posterior, theta_grid)   # posterior mean
    theta_map = theta_grid[np.argmax(posterior)]                     # posterior mode

    bayes_history.append(theta_bayes)
    map_history.append(theta_map)

steps = np.arange(1, n_items + 1)

# ---- visualization -----------------------------------------------------
fig7 = go.Figure()
fig7.add_trace(go.Scatter(x=steps, y=bayes_history, mode="lines+markers",
                           name="Posterior Mean (Bayes)"))
fig7.add_trace(go.Scatter(x=steps, y=map_history, mode="lines+markers",
                           name="MAP estimate"))
fig7.add_hline(y=theta_true, line_dash="dash", line_color="red",
               annotation_text="theta_true = 0.75", annotation_position="bottom right")

fig7.update_layout(
    title=f"Convergence of Running Ability Estimators over {n_items} Items",
    xaxis_title="Item step k",
    yaxis_title="Estimated theta",
    template="plotly_white",
    width=850, height=550
)
fig7.show()
# fig7.write_html("task7_convergence.html")

print("Final Bayes estimate:", bayes_history[-1])
print("Final MAP estimate:  ", map_history[-1])

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

## Bayesian Tracking of Click-Through Rates via Conjugate Beta-Binomial Updates — Solution

### Task 1: Structural Probability and Properties

Plotted $f(\theta;\alpha,\beta)=\dfrac{1}{B(\alpha,\beta)}\theta^{\alpha-1}(1-\theta)^{\beta-1}$ for $(\alpha,\beta)\in\{(1,1),(2,8),(8,2)\}$ (see accompanying code).

**Interpretation.** $\alpha$ and $\beta$ act as pseudo-counts of "successes" and "failures" respectively. The mean of a Beta distribution is $\dfrac{\alpha}{\alpha+\beta}$, so the balance between $\alpha$ and $\beta$ directly sets where the density's mass concentrates on $[0,1]$:
- $\alpha=\beta$ (e.g. $1,1$) $\Rightarrow$ symmetric, centered at $0.5$ — here flat/uninformative.
- $\alpha<\beta$ (e.g. $2,8$) $\Rightarrow$ mass concentrated toward $0$, right-skewed tail — reflects a belief in a low click rate.
- $\alpha>\beta$ (e.g. $8,2$) $\Rightarrow$ mass concentrated toward $1$, left-skewed tail — reflects a belief in a high click rate.

Increasing $\alpha+\beta$ while holding the ratio fixed narrows the density (more concentrated belief) without moving its center.

---

### Task 2: Sequential Likelihood and Joint History

A single isolated response $y_k$ is Bernoulli($\theta$), so

$$
L(y_k \mid \theta) = \theta^{y_k}(1-\theta)^{1-y_k}.
$$

Under conditional independence of impressions given $\theta$, the joint likelihood of the running history $\mathbf{y}^{(k)}=(y_1,\dots,y_k)$ is

$$
L\big(\mathbf{y}^{(k)} \mid \theta\big) = \prod_{i=1}^{k}\theta^{y_i}(1-\theta)^{1-y_i} = \theta^{S_k}(1-\theta)^{k-S_k},
\qquad S_k=\sum_{i=1}^{k} y_i,
$$

i.e. it depends on the history only through the total click count $S_k$ — a sufficient statistic.

---

### Task 3: Closed-Form Analytical Updates (Conjugacy)

By Bayes' Theorem, using the previous posterior as the new prior:

$$
f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta\mid\mathbf{y}^{(k)})
\;\propto\;
f_{\Theta\mid\mathbf{Y}^{(k-1)}}(\theta\mid\mathbf{y}^{(k-1)})\cdot L(y_k\mid\theta).
$$

Substituting the Beta form $f_{\Theta\mid\mathbf{Y}^{(k-1)}}(\theta)\propto \theta^{\alpha_{k-1}-1}(1-\theta)^{\beta_{k-1}-1}$ and $L(y_k\mid\theta)=\theta^{y_k}(1-\theta)^{1-y_k}$:

$$
f_{\Theta\mid\mathbf{Y}^{(k)}}(\theta)
\;\propto\;
\theta^{\,\alpha_{k-1}+y_k-1}\,(1-\theta)^{\,\beta_{k-1}+(1-y_k)-1}.
$$

This is (up to normalization) exactly the kernel of a $\text{Beta}(\alpha_k,\beta_k)$ density with

$$
\boxed{\alpha_k = \alpha_{k-1} + y_k, \qquad \beta_k = \beta_{k-1} + (1-y_k).}
$$

Since a Beta prior combined with a Bernoulli likelihood yields another Beta posterior, the family is **self-conjugate** (Beta–Binomial conjugacy): no numerical normalization or integration is ever required — every update is a simple increment of one of the two parameters, depending on whether $y_k=1$ or $y_k=0$.

**Posterior mean at step $k$:**

$$
\mathbb{E}\big[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}\big] = \frac{\alpha_k}{\alpha_k+\beta_k}.
$$

---

### Task 4: Dynamic Shifting Mechanics

Each update adds exactly $1$ to either $\alpha_k$ (if $y_k=1$) or $\beta_k$ (if $y_k=0$):

- **Click ($y_k=1$):** $\alpha_k=\alpha_{k-1}+1$, $\beta_k=\beta_{k-1}$. Since the mode of $\text{Beta}(\alpha,\beta)$ (for $\alpha,\beta>1$) is $\dfrac{\alpha-1}{\alpha+\beta-2}$, increasing $\alpha$ alone increases this ratio — the peak of the density shifts **right**, toward higher believed CTR.
- **No click ($y_k=0$):** $\beta_k=\beta_{k-1}+1$, $\alpha_k=\alpha_{k-1}$. This decreases the mode ratio — the peak shifts **left**, toward lower believed CTR.

The magnitude of each shift naturally shrinks as $\alpha_k+\beta_k$ grows, since a single new count matters less relative to a larger accumulated total (the update is self-damping).

**Contrast with non-conjugate models (e.g. 2PL IRT).** In the 2PL setting the likelihood factor $p_k(\theta)^{y_k}(1-p_k(\theta))^{1-y_k}$ is a logistic function of $\theta$, not a polynomial in $\theta$ and $(1-\theta)$ — multiplying it by a Gaussian (or any) prior does **not** stay within a closed parametric family. That forces the posterior to be tracked and renormalized numerically on a discretized grid at every step. Here, by contrast, the entire running posterior is captured by just two scalars $(\alpha_k,\beta_k)$ updated by simple arithmetic — no grid, no numerical integration, and no approximation error at any step.

---

### Task 5: Running Point Estimators

From the updated shape parameters $\alpha_k,\beta_k$ at step $k$:

**Running Posterior Mean:**

$$
\hat\theta_{\text{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k+\beta_k}.
$$

**Running Maximum A Posteriori (mode of $\text{Beta}(\alpha_k,\beta_k)$):**

$$
\hat\theta_{\text{MAP}}^{(k)} =
\begin{cases}
\dfrac{\alpha_k-1}{\alpha_k+\beta_k-2}, & \alpha_k>1,\ \beta_k>1 \\[2mm]
0, & \alpha_k\le 1,\ \beta_k>1 \\[1mm]
1, & \beta_k\le 1,\ \alpha_k>1 \\[1mm]
\text{any } \theta\in[0,1] \text{ (flat)}, & \alpha_k=\beta_k=1
\end{cases}
$$

(With $\alpha_0=\beta_0=1$ and $+1$ increments each step, $\alpha_k,\beta_k\ge 1$ always, so only the first and last cases arise once $k\ge1$.)

---

### Task 6: Performance Tracking and Convergence Analysis

See the accompanying Python code for the full simulation ($\theta_{\text{true}}=0.35$, $\alpha_0=\beta_0=1$, $n=100$ impressions, closed-form updates at every step).

**Analysis.** As $k\to100$, both $\hat\theta_{\text{Bayes}}^{(k)}$ and $\hat\theta_{\text{MAP}}^{(k)}$ converge toward $\theta_{\text{true}}=0.35$, and the gap between the two estimators shrinks together with them (they coincide once $\alpha_k,\beta_k$ are large, since mean and mode of a Beta distribution converge to the same value as its variance $\dfrac{\alpha\beta}{(\alpha+\beta)^2(\alpha+\beta+1)}$ shrinks). Early on, the choice of prior ($\alpha_0=\beta_0=1$, i.e., uniform) has a visible effect — the first few impressions can swing the estimate noticeably. As $k$ grows, $\alpha_k+\beta_k\approx k$ dominates the denominator in both formulas, so the accumulated data increasingly outweighs the initial pseudo-counts from the prior. This is the general Bayesian principle that **the influence of the prior is asymptotically washed out by data**: with enough impressions, the posterior estimate is governed almost entirely by the observed click history rather than by the initial assumption, and the platform's estimate becomes both more accurate and less sensitive to any single new impression.


In [1]:
"""
Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates
Python code (Tasks 1 and 6) — run cells in Colab, or execute as a script.
Requires: pip install plotly numpy scipy
"""

import numpy as np
from scipy.stats import beta as beta_dist
import plotly.graph_objects as go

# =====================================================================
# TASK 1 — Structural Probability and Properties: Beta(alpha, beta) PDFs
# =====================================================================

theta = np.linspace(0, 1, 500)

param_sets = [
    (1, 1, "Uninformative: (a=1, b=1)"),
    (2, 8, "Right-skewed: (a=2, b=8)"),
    (8, 2, "Left-skewed: (a=8, b=2)"),
]

fig1 = go.Figure()
for a, b, label in param_sets:
    pdf = beta_dist.pdf(theta, a, b)
    fig1.add_trace(go.Scatter(x=theta, y=pdf, mode="lines", name=label))

fig1.update_layout(
    title="Beta(alpha, beta) Probability Density Functions",
    xaxis_title="theta",
    yaxis_title="Density f(theta)",
    template="plotly_white",
    width=850, height=550
)
fig1.show()
# fig1.write_html("task1_beta_pdfs.html")


# =====================================================================
# TASK 6 — Performance Tracking and Convergence Analysis
# =====================================================================

rng = np.random.default_rng(42)

theta_true = 0.35
n_impressions = 100

alpha_0, beta_0 = 1.0, 1.0   # uniform initial prior

alpha_k, beta_k = alpha_0, beta_0

# step 0 estimators (uniform prior -> mean 0.5, mode undefined/flat -> use 0.5)
bayes_history = [alpha_k / (alpha_k + beta_k)]
map_history = [0.5]

for k in range(1, n_impressions + 1):
    # simulate response by comparing a uniform draw against theta_true
    u = rng.uniform(0.0, 1.0)
    y_k = 1 if u < theta_true else 0

    # closed-form conjugate update
    alpha_k = alpha_k + y_k
    beta_k = beta_k + (1 - y_k)

    # running posterior mean
    theta_bayes = alpha_k / (alpha_k + beta_k)

    # running MAP (mode of Beta(alpha_k, beta_k))
    if alpha_k > 1 and beta_k > 1:
        theta_map = (alpha_k - 1) / (alpha_k + beta_k - 2)
    elif alpha_k <= 1 and beta_k > 1:
        theta_map = 0.0
    elif beta_k <= 1 and alpha_k > 1:
        theta_map = 1.0
    else:
        theta_map = 0.5  # alpha_k == beta_k == 1, flat density

    bayes_history.append(theta_bayes)
    map_history.append(theta_map)

steps = np.arange(0, n_impressions + 1)

# ---- visualization -----------------------------------------------------
fig6 = go.Figure()
fig6.add_trace(go.Scatter(x=steps, y=bayes_history, mode="lines",
                           name="Posterior Mean (Bayes)"))
fig6.add_trace(go.Scatter(x=steps, y=map_history, mode="lines",
                           name="MAP estimate"))
fig6.add_hline(y=theta_true, line_dash="dash", line_color="red",
               annotation_text="theta_true = 0.35", annotation_position="bottom right")

fig6.update_layout(
    title=f"Convergence of Running CTR Estimators over {n_impressions} Impressions",
    xaxis_title="Impression step k",
    yaxis_title="Estimated theta",
    template="plotly_white",
    width=850, height=550
)
fig6.show()
# fig6.write_html("task6_convergence.html")

print("Final alpha_k, beta_k:", alpha_k, beta_k)
print("Final Bayes estimate:", bayes_history[-1])
print("Final MAP estimate:  ", map_history[-1])

Final alpha_k, beta_k: 34.0 68.0
Final Bayes estimate: 0.3333333333333333
Final MAP estimate:   0.33


# Q Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates

In aerospace and civil engineering, Structural Health Monitoring (SHM) is critical for detecting damage before a catastrophic failure occurs. Consider an aircraft wing or a bridge girder equipped with specialized vibration sensors. Over time, environmental fatigue or dynamic impacts can cause micro-fractures, resulting in a reduction of the component's mechanical stiffness.

Let $\Theta = \theta$ represent the structural **remaining stiffness efficiency factor**, where $\theta$ is physically bounded to the interval:

$$\theta \in (0, 1]$$

* $\theta = 1.0$ indicates a perfectly pristine, undamaged structural component.
* $\theta \to 0$ signifies critical degradation or severe structural cracking.

Let $K_{\text{nominal}}$ be the known, baseline stiffness of the structural component when it is entirely healthy. At each sequential inspection time step $k$ (where $k = 1, 2, \dots, n$), a sensor collects a noisy experimental stiffness measurement $y_k$.

Engineers model the degradation physics via a non-linear relationship with multiplicative log-normal measurement noise to prevent non-physical negative values:

$$y_k = \theta \cdot K_{\text{nominal}} \cdot e^{\epsilon_k}, \qquad \epsilon_k \sim \mathscr{N}(0, \sigma^2)$$

where $\sigma$ is the standard deviation of the sensor noise in log-space.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running history vector of observed sensor readings** up to the current inspection milestone. Before deploying the sensors, engineers utilize an initial prior distribution $f_{\Theta}^{(0)}(\theta)$ over the domain $(0, 1]$ based on historical manufacturing specifications. As the sensor stream arrives, the posterior distribution calculated at step $k-1$ serves directly as the prior distribution for step $k$.

---

### **Tasks**

#### **1. Prior Belief Boundaries**

Before data collection begins, engineers assume the component is highly likely to be healthy, modeling this using a bounded Beta distribution as the initial prior: $\Theta \sim \text{Beta}(8, 1.5)$.

* Plot this initial prior density function using Plotly over the restricted physical domain $\theta \in [0.01, 1.0]$.
* Calculate the expected prior stiffness efficiency $\mathbb{E}[\Theta^{(0)}]$ analytically. Explain why this specific distribution serves as an appropriate initial prior for an engineering component assumed to be healthy.

#### **2. Structural Likelihood Formulation**

Using the change of variables or properties of the log-normal distribution, write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* continuous sensor measurement $y_k$ at inspection step $k$, given the true stiffness factor $\theta$. Following this, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

#### **3. Mathematical Formulation of the Non-Conjugate Grid Update**

Explain why an exact closed-form analytical solution for the posterior density $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$ does not exist when combining a Beta prior with this log-normal structural likelihood. Write down the recursive relationship for the posterior density at step $k$ up to a proportionality constant.

#### **4. Running Point Estimates**

Because a closed-form formula is unavailable, we must define point estimators through numerical integration. Write down the definite integral equations over the bounded domain $(0, 1]$ required to compute:

* The **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* The **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

#### **5. Algorithmic Grid Approximation and Normalization**

Describe the step-by-step numerical procedure to maintain this distribution on a discrete grid of $\theta$-values. Explicitly state how you would handle the boundary limits computationally and how you would perform the sequential normalization step using the trapezoidal rule after a new sensor reading $y_k$ is observed.

#### **6. Performance Tracking and Degradation Convergence Analysis**

Suppose an impact occurs, and the true, hidden remaining stiffness drops to $\theta_{\text{true}} = 0.68$. Write a Python script using Plotly to simulate an engineered monitoring timeline across $n = 15$ continuous sensor measurements ($K_{\text{nominal}} = 50.0 \text{ kN/mm}$, $\sigma = 0.15$):

* **Simulate Sensor Stream:** Programmatically generate noisy sensor readings $y_k$ by drawing random values from the underlying log-normal physics model centered at $\theta_{\text{true}}$.
* **Track Estimators:** Loop sequentially through each step. At each step, update the unnormalized grid, normalize it via `np.trapezoid`, and compute both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize Curves & Timeline:** Generate two plots:
1. A plot showing the progression of the full posterior density curves at milestones $k \in \{0, 1, 2, 5, 10, 15\}$.
2. A line chart tracking the convergence of both $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$ from step $0$ to $15$ against a horizontal reference line at $\theta_{\text{true}} = 0.68$.


* **Analysis:** Evaluate the behavior of the distribution. How many sensor readings did it take for the system to overcome the initially optimistic "healthy" prior and confidently isolate the 68% damage state? What does the narrowing of the density curves imply about structural safety thresholds?

## Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates — Solution

### Task 1: Prior Belief Boundaries

Plotted the initial prior $\Theta \sim \text{Beta}(8, 1.5)$ over $\theta\in[0.01,1.0]$ (see accompanying code).

**Expected prior stiffness efficiency:**

$$
\mathbb{E}\big[\Theta^{(0)}\big] = \frac{\alpha_0}{\alpha_0+\beta_0} = \frac{8}{8+1.5} = \frac{8}{9.5} \approx 0.842.
$$

**Why this prior is appropriate.** $\text{Beta}(8,1.5)$ has its mode at $\dfrac{\alpha_0-1}{\alpha_0+\beta_0-2}=\dfrac{7}{7.5}\approx0.933$, is naturally bounded on $(0,1)$ (matching the physical constraint that stiffness efficiency cannot exceed $1$ or fall below $0$), and places the bulk of its density mass close to $\theta=1$ with a rapidly vanishing left tail near $\theta=0$. This encodes engineers' prior belief — based on manufacturing specs — that a newly deployed component is almost certainly close to pristine, while still allowing (with low but nonzero probability) for an already-degraded state.

---

### Task 2: Structural Likelihood Formulation

Given $\theta$, the sensor model is $y_k = \theta K_{\text{nominal}} e^{\epsilon_k}$ with $\epsilon_k\sim\mathcal N(0,\sigma^2)$, so $\epsilon_k = \ln y_k - \ln(\theta K_{\text{nominal}})$. Changing variables from $\epsilon_k$ to $y_k$ (Jacobian $|d\epsilon_k/dy_k| = 1/y_k$) gives $y_k\mid\theta$ a **log-normal** density:

$$
L(y_k\mid\theta) = \frac{1}{y_k\,\sigma\sqrt{2\pi}}\exp\!\left(-\frac{\big[\ln y_k - \ln(\theta K_{\text{nominal}})\big]^2}{2\sigma^2}\right).
$$

Assuming conditional independence of sensor readings given $\theta$, the joint likelihood for the running history $\mathbf{y}^{(k)}=(y_1,\dots,y_k)$ is

$$
L\big(\mathbf{y}^{(k)}\mid\theta\big) = \prod_{i=1}^{k}\frac{1}{y_i\,\sigma\sqrt{2\pi}}\exp\!\left(-\frac{\big[\ln y_i - \ln(\theta K_{\text{nominal}})\big]^2}{2\sigma^2}\right).
$$

---

### Task 3: Mathematical Formulation of the Non-Conjugate Grid Update

**Why no closed form exists.** The Beta prior has kernel $\theta^{\alpha-1}(1-\theta)^{\beta-1}$ — a polynomial-type function of $\theta$. The log-normal likelihood's kernel, $\exp\!\big(-(\ln\theta - c)^2/2\sigma^2\big)$, is a Gaussian bump in $\ln\theta$, not in $\theta$ itself. Multiplying $\theta^{\alpha-1}(1-\theta)^{\beta-1}$ by $\exp(-(\ln\theta-c)^2/2\sigma^2)$ does **not** collapse back into a Beta kernel, nor into any other standard named family — there is no algebraic identity that re-expresses this product as a recognized distribution with a simple normalizing constant. The Beta prior and log-normal likelihood are therefore **non-conjugate**, and the posterior has no closed-form update rule (unlike the Beta–Binomial case).

**Recursive relationship (up to proportionality):**

$$
f_{\Theta\mid\mathbf{Y}^{(k)}}\big(\theta\mid\mathbf{y}^{(k)}\big)
\;\propto\;
f_{\Theta\mid\mathbf{Y}^{(k-1)}}\big(\theta\mid\mathbf{y}^{(k-1)}\big)\cdot L(y_k\mid\theta),
\qquad \theta\in(0,1).
$$

---

### Task 4: Running Point Estimates

Since there is no closed form, both estimators must be obtained by numerically integrating the (normalized) posterior over the bounded domain $(0,1)$:

**Running Posterior Mean:**

$$
\hat\theta_{\text{Bayes}}^{(k)} = \int_0^1 \theta\, f_{\Theta\mid\mathbf{Y}^{(k)}}\big(\theta \mid \mathbf{y}^{(k)}\big)\, d\theta.
$$

**Running Maximum A Posteriori:**

$$
\hat\theta_{\text{MAP}}^{(k)} = \arg\max_{\theta\in(0,1)} f_{\Theta\mid\mathbf{Y}^{(k)}}\big(\theta \mid \mathbf{y}^{(k)}\big).
$$

(The MAP requires only locating the maximizing grid point, not integration; the posterior mean requires the definite integral above.)

---

### Task 5: Algorithmic Grid Approximation and Normalization

1. **Discretize.** Choose a fine grid of $\theta$-values over the physical domain, e.g. $\theta_1,\dots,\theta_M$ evenly spaced in $[\epsilon, 1]$ for a small $\epsilon>0$ (e.g. $0.001$), since $\theta=0$ is a singular point ($\ln\theta\to-\infty$ makes the log-normal likelihood ill-defined there) and $\theta=1$ is the natural physical ceiling (fully healthy component).
2. **Initialize.** Evaluate the $\text{Beta}(8,1.5)$ prior on the grid: $f^{(0)}[\theta_m]$.
3. **Per-reading update.** After observing a new sensor value $y_k$:
   - Compute $L(y_k\mid\theta_m)$ for every grid point using the log-normal formula from Task 2.
   - Form the unnormalized product $g[\theta_m] = f^{(k-1)}[\theta_m]\cdot L(y_k\mid\theta_m)$.
4. **Sequential normalization (trapezoidal rule).** Numerically integrate $g$ over the grid, $Z_k = \int_\epsilon^1 g(\theta)\,d\theta \approx \text{trapz}(g,\theta_{\text{grid}})$, then set $f^{(k)}[\theta_m] = g[\theta_m]/Z_k$. This both normalizes the posterior and implicitly re-imposes the bounded domain, since any probability mass outside $[\epsilon,1]$ is excluded by construction (truncation to the physical support at every step).
5. **Iterate.** $f^{(k)}$ becomes the prior grid for step $k+1$; repeat steps 3–4 for each new reading.
6. **Extract estimators** at any step via $\hat\theta_{\text{Bayes}}^{(k)} = \sum_m \theta_m f^{(k)}[\theta_m]\,\Delta\theta$ (trapezoidal sum) and $\hat\theta_{\text{MAP}}^{(k)} = \theta_{m^\*}$ where $m^\* = \arg\max_m f^{(k)}[\theta_m]$.

---

### Task 6: Performance Tracking and Degradation Convergence Analysis

See the accompanying Python code for the full simulation ($\theta_{\text{true}}=0.68$, $K_{\text{nominal}}=50.0$ kN/mm, $\sigma=0.15$, $n=15$ readings, prior $\text{Beta}(8,1.5)$), including the posterior-curve snapshots at $k\in\{0,1,2,5,10,15\}$ and the convergence line chart for both estimators.

**Analysis.** The optimistic prior starts both estimators high ($\hat\theta^{(0)}_{\text{Bayes}}\approx0.84$, $\hat\theta^{(0)}_{\text{MAP}}\approx0.93$). The first one or two readings, being far below what a healthy component would produce, already pull both estimators sharply downward; by around $k=5$–$6$ readings the estimators have largely overcome the "healthy" prior and settled within a narrow band around the true value of $0.68$, after which additional readings produce only small refinements. Correspondingly, the posterior density curves at the later milestones ($k=10, 15$) are visibly **narrower and taller**, centered tightly on $\theta\approx0.68$, compared to the wide, right-shifted prior curve at $k=0$.

This narrowing has a direct engineering implication: initially the monitoring system cannot yet distinguish a genuinely damaged component from ordinary sensor noise around a healthy prior, but after enough independent readings accumulate, the posterior becomes sharply peaked and the system can **confidently isolate** the $68\%$ stiffness-efficiency degradation state well above the noise floor. In a real SHM deployment, this narrowing posterior is exactly what would justify triggering a maintenance/safety alert threshold — the system needs sufficient evidence (here, roughly $5$–$6$ readings) before it can commit to flagging a genuine structural degradation rather than reacting to a single noisy measurement.


In [2]:
"""
Bayesian Estimations for Structural Health Monitoring via Bounded Grid Updates
Python code (Tasks 1 and 6) — run cells in Colab, or execute as a script.
Requires: pip install plotly numpy scipy
"""

import numpy as np
from scipy.stats import beta as beta_dist
import plotly.graph_objects as go

# =====================================================================
# TASK 1 — Prior Belief Boundaries: Beta(8, 1.5) initial prior
# =====================================================================

alpha_0, beta_0 = 8.0, 1.5
theta_plot = np.linspace(0.01, 1.0, 500)
prior_pdf = beta_dist.pdf(theta_plot, alpha_0, beta_0)

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=theta_plot, y=prior_pdf, mode="lines",
                           name=f"Beta({alpha_0}, {beta_0}) prior"))
fig1.update_layout(
    title="Initial Prior: Remaining Stiffness Efficiency Θ ~ Beta(8, 1.5)",
    xaxis_title="theta (stiffness efficiency)",
    yaxis_title="Density f(theta)",
    template="plotly_white",
    width=850, height=550
)
fig1.show()
# fig1.write_html("task1_shm_prior.html")

E_theta0 = alpha_0 / (alpha_0 + beta_0)
print(f"E[Theta^(0)] = alpha_0/(alpha_0+beta_0) = {E_theta0:.4f}")


# =====================================================================
# TASK 6 — Performance Tracking and Degradation Convergence Analysis
# =====================================================================

rng = np.random.default_rng(42)

theta_true = 0.68
K_nominal = 50.0     # kN/mm
sigma = 0.15
n_steps = 15

# grid over the physical domain; avoid exact 0 since ln(theta) is used
theta_grid = np.linspace(0.001, 1.0, 1000)

def lognormal_likelihood(y, theta, K_nominal, sigma):
    """Density of y | theta under y = theta*K_nominal*exp(epsilon), epsilon~N(0,sigma^2)."""
    mu = np.log(theta * K_nominal)
    return (1.0 / (y * sigma * np.sqrt(2 * np.pi))) * \
           np.exp(-(np.log(y) - mu) ** 2 / (2 * sigma ** 2))

# initialize posterior = prior on the grid
posterior = beta_dist.pdf(theta_grid, alpha_0, beta_0)
posterior /= np.trapezoid(posterior, theta_grid)

bayes_history = [np.trapezoid(theta_grid * posterior, theta_grid)]
map_history = [theta_grid[np.argmax(posterior)]]
snapshot_steps = {0, 1, 2, 5, 10, 15}
snapshots = {0: posterior.copy()}

for k in range(1, n_steps + 1):
    # simulate a noisy sensor reading at the true stiffness
    epsilon_k = rng.normal(0.0, sigma)
    y_k = theta_true * K_nominal * np.exp(epsilon_k)

    # likelihood of y_k across the grid
    likelihood = lognormal_likelihood(y_k, theta_grid, K_nominal, sigma)

    # unnormalized posterior, then normalize via trapezoidal rule
    unnormalized = posterior * likelihood
    posterior = unnormalized / np.trapezoid(unnormalized, theta_grid)

    theta_bayes = np.trapezoid(theta_grid * posterior, theta_grid)
    theta_map = theta_grid[np.argmax(posterior)]

    bayes_history.append(theta_bayes)
    map_history.append(theta_map)

    if k in snapshot_steps:
        snapshots[k] = posterior.copy()

steps = np.arange(0, n_steps + 1)

# ---- Plot 1: posterior density curves at milestones ---------------------
fig6a = go.Figure()
for k in sorted(snapshots.keys()):
    fig6a.add_trace(go.Scatter(x=theta_grid, y=snapshots[k], mode="lines",
                                name=f"k={k}"))
fig6a.add_vline(x=theta_true, line_dash="dash", line_color="red",
                annotation_text="theta_true = 0.68")
fig6a.update_layout(
    title="Posterior Density Progression at Inspection Milestones",
    xaxis_title="theta (stiffness efficiency)",
    yaxis_title="Density f(theta | y^(k))",
    template="plotly_white",
    width=850, height=550
)
fig6a.show()
# fig6a.write_html("task6_posterior_curves.html")

# ---- Plot 2: convergence of running estimators ---------------------------
fig6b = go.Figure()
fig6b.add_trace(go.Scatter(x=steps, y=bayes_history, mode="lines+markers",
                            name="Posterior Mean (Bayes)"))
fig6b.add_trace(go.Scatter(x=steps, y=map_history, mode="lines+markers",
                            name="MAP estimate"))
fig6b.add_hline(y=theta_true, line_dash="dash", line_color="red",
                annotation_text="theta_true = 0.68", annotation_position="bottom right")
fig6b.update_layout(
    title=f"Convergence of Running Stiffness Estimators over {n_steps} Sensor Readings",
    xaxis_title="Inspection step k",
    yaxis_title="Estimated theta",
    template="plotly_white",
    width=850, height=550
)
fig6b.show()
# fig6b.write_html("task6_convergence.html")

print("Bayes estimate history:", [round(v, 4) for v in bayes_history])
print("MAP estimate history:  ", [round(v, 4) for v in map_history])

E[Theta^(0)] = alpha_0/(alpha_0+beta_0) = 0.8421


Bayes estimate history: [np.float64(0.8421), np.float64(0.7947), np.float64(0.6977), np.float64(0.7177), np.float64(0.7332), np.float64(0.6823), np.float64(0.6604), np.float64(0.6649), np.float64(0.6629), np.float64(0.6645), np.float64(0.6577), np.float64(0.6676), np.float64(0.6751), np.float64(0.676), np.float64(0.6844), np.float64(0.6873)]
MAP estimate history:   [np.float64(0.933), np.float64(0.797), np.float64(0.688), np.float64(0.711), np.float64(0.728), np.float64(0.678), np.float64(0.657), np.float64(0.662), np.float64(0.66), np.float64(0.662), np.float64(0.655), np.float64(0.666), np.float64(0.673), np.float64(0.674), np.float64(0.683), np.float64(0.686)]


# Q. Gaussian Mixture Clustering as Conditional Updating

Consider a dataset
$$
x_1,x_2,\dots,x_n\in\mathbb R^d.
$$
We wish to cluster these observations into $K$ groups. Instead of assigning each point deterministically to a cluster at the beginning, we introduce a latent random variable
$$
C_i\in{1,\dots,K},
$$
where $C_i=k$ means that the observation $x_i$ belongs to cluster $k$.
Let the prior probability of cluster membership be
$$
P(C_i=k)=\phi_k,
$$
where
$$
\phi_k\ge 0,
\qquad
\sum_{k=1}^K \phi_k=1.
$$

Conditional on $C_i=k$, assume that the observation $X_i$ is generated from a multivariate Gaussian distribution:
$$
X_i\mid C_i=k
\sim
\mathscr N(\mu_k,\Sigma_k),
$$
where
$$
\mu_k\in\mathbb R^d,
\qquad
\Sigma_k\in\mathbb R^{d\times d}
$$
are the mean vector and covariance matrix of cluster $k$.

The model parameters
$$
\phi_k,\mu_k,\Sigma_k,
\qquad k=1,\dots,K,
$$
are assumed to be fixed but unknown.

---

1. Deriving the Marginal Density:
Using the law of total probability, show that the marginal density of $X_i$ is
$$
p(x_i)=\sum_{k=1}^K
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k).
$$
Explain why this density is called a Gaussian mixture density.

---

2. Deriving the Posterior Cluster Probability:
For a fixed observation $x_i$, use Bayes' rule to derive
$$
P(C_i=k\mid X_i=x_i)=\frac{
P(X_i=x_i\mid C_i=k)P(C_i=k)
}{
\sum_{j=1}^K P(X_i=x_i\mid C_i=j)P(C_i=j)
}.
$$
Then substitute the Gaussian model and the cluster prior to obtain
$$
P(C_i=k\mid X_i=x_i)=\frac{
\phi_k\mathscr N(x_i\mid \mu_k,\Sigma_k)
}{
\sum_{j=1}^K
\phi_j\mathscr N(x_i\mid \mu_j,\Sigma_j)
}.
$$
This quantity is called the responsibility of cluster $k$ for data point $x_i$, and is denoted by
$$
\gamma_{ik}=P(C_i=k\mid X_i=x_i).
$$
Explain why $\gamma_{ik}$ may be interpreted as a posterior probability of cluster membership.

---

3. One-Hot Encoding of the Latent Cluster Variable:
Now define a one-hot encoded latent random vector
$$
Z_i=
\begin{bmatrix}
Z_{i1}\\
Z_{i2}\\
\vdots\\
Z_{iK}
\end{bmatrix},
$$
where
$$
Z_{ik}=\begin{cases}
1, & \text{if } C_i=k,\\
0, & \text{otherwise}.
\end{cases}
$$
Show that
$$
\mathbb E[Z_{ik}\mid X_i=x_i]=P(C_i=k\mid X_i=x_i).
$$
Hence show that
$$
\mathbb E[Z_i\mid X_i=x_i]=\begin{bmatrix}
\gamma_{i1}\\
\gamma_{i2}\\
\vdots\\
\gamma_{iK}
\end{bmatrix}.
$$
Conclude that the soft cluster assignment in a Gaussian mixture model is precisely the conditional expectation
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

---

4. From Soft Assignment to Hard Clustering:
The vector
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
gives a soft assignment of $x_i$ to all clusters. A hard cluster assignment can be obtained by choosing the cluster with the largest posterior probability:
$$
\widehat C_i=\operatorname{arg\,max}_{1\le k\le K}
\gamma_{ik}.
$$
Explain the difference between soft clustering and hard clustering in this context.

---

5. Conditional Expectation of the Observation Given the Cluster:
Show that
$$
\mathbb E[X_i\mid C_i=k]=\mu_k.
$$
Explain why $\mu_k$ can be interpreted as the center of cluster $k$.
Then compare the two conditional expectations
$$
\mathbb E[Z_i\mid X_i=x_i]
$$
and
$$
\mathbb E[X_i\mid C_i=k].
$$
Explain why the first gives the soft cluster membership of an observed point, while the second gives the mean location of a cluster.

---

6. The Complete-Data Likelihood
If the latent labels $z_i$ were known, the complete-data likelihood would be
$$
p(x_1,\dots,x_n,z_1,\dots,z_n)=\prod_{i=1}^n
\prod_{k=1}^K
\left[
\phi_k
\mathscr N(x_i\mid \mu_k,\Sigma_k)
\right]^{z_{ik}}.
$$
Take the logarithm and show that the complete-data log-likelihood is
$$
\ell_c=\sum_{i=1}^n
\sum_{k=1}^K
z_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why this expression would be easy to maximize if the values of $z_{ik}$ were known.

---

7. The EM Interpretation:
In practice, the latent variables $Z_i$ are not observed. The EM algorithm replaces the unknown indicators $z_{ik}$ by their conditional expectations given the observed data and current parameter estimates:
$$
z_{ik}
\quad\leadsto\quad
\mathbb E[Z_{ik}\mid X_i=x_i].
$$
That is,
$$
z_{ik}
\quad\leadsto\quad
\gamma_{ik}.
$$
This is the E-step of the EM algorithm.
Using this idea, write the expected complete-data log-likelihood:
$$
Q=\sum_{i=1}^n
\sum_{k=1}^K
\gamma_{ik}
\left[
\log \phi_k
+
\log \mathscr N(x_i\mid \mu_k,\Sigma_k)
\right].
$$
Explain why the E-step can be interpreted as a conditional update of cluster membership probabilities.

---

8. Parameter Updates:
By maximizing $Q$ with respect to the model parameters, derive the standard GMM updates:
$$
N_k=\sum_{i=1}^n \gamma_{ik},
$$
$$
\phi_k^{\text{new}}=\frac{N_k}{n},
$$
$$
\mu_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}x_i,
$$
and
$$
\Sigma_k^{\text{new}}=\frac{1}{N_k}
\sum_{i=1}^n
\gamma_{ik}
(x_i-\mu_k^{\text{new}})
(x_i-\mu_k^{\text{new}})^T.
$$
Explain how the responsibility $\gamma_{ik}$ acts as a fractional membership weight of observation $x_i$ in cluster $k$.

---

9. Interpretation:
Write a short paragraph explaining why GMM clustering can be viewed as a repeated process of conditional updating.
Your answer should mention the following points:

* The mixture weight $\phi_k$ is the prior probability of cluster $k$.
* The Gaussian density $\mathscr N(x_i\mid \mu_k,\Sigma_k)$ measures how compatible $x_i$ is with cluster $k$.
* The responsibility $\gamma_{ik}$ is the posterior probability of cluster $k$ after observing $x_i$.
* The soft assignment vector is
$$
\mathbb E[Z_i\mid X_i=x_i].
$$

* The M-step updates the cluster parameters using these posterior membership probabilities as weights.
Conclude that Gaussian mixture clustering is probabilistic clustering based on conditional expectations of latent cluster membership variables.

---

Here is a perfectly tailored question that you can add as the final part (**Part 10**) of your assignment notebook to bridge your theoretical derivations with your code implementation:

---

10. Computational Simulation and Out-of-Sample Validation

Using the theoretical framework established in the previous parts, write a Python class named `GMMFinancialSegmenter` that implements a two-dimensional Gaussian Mixture Model (GMM) using `scikit-learn` and visualizes the results interactively using `Plotly`. Your implementation should fulfill the following criteria:

* **Data Splitting and Scaling:** Accept a dataset containing two continuous features (e.g., mimicking financial behaviors like `PURCHASES` and `CREDIT_LIMIT`), standardize the features to handle variance scaling, and split the data into an 80% training set and a 20% validation/test set.
* **EM Execution:** Fit a GMM with $K=3$ components on the training data using the Expectation-Maximization (EM) algorithm, printing whether the model successfully converged and the number of iterations required.
* **Out-of-Sample Performance:** Compute and output the average log-likelihood score over the unseen test set to validate how well the learned density functions generalize to new data.
* **Interactive Visualizations:** Implement methods to generate three distinct Plotly figures:
1. An empirical **2D Density Heatmap** of the raw training data with marginal distributions to inspect its underlying multimodal structure.
2. A **Training Assignment Plot** that overlays the training data points on top of a continuous contour map showing the maximum posterior responsibilities ($\gamma_{ik}$) computed across a fine coordinate grid.
3. A **Test Assignment Plot** that replicates the contour boundary visualization but overlays out-of-sample test data points to expose the physical regions of cluster ambiguity.



Briefly evaluate the resulting plots. Explain how the continuous background contour map visually demonstrates the soft assignment expectation vector $\mathbb{E}[Z_i \mid X_i = x_{\text{grid}}]$ that you proved analytically in Part 3.

Use the dataset

https://www.kaggle.com/datasets/arjunbhasin2013/ccdata

## Gaussian Mixture Models: Latent-Variable Derivation and EM — Solution

### Part 1: Deriving the Marginal Density

By the law of total probability, conditioning on the discrete latent label $C_i$:

$$
p(x_i) = \sum_{k=1}^{K} P(C_i=k)\, p(x_i \mid C_i=k) = \sum_{k=1}^{K} \phi_k\, \mathcal N(x_i \mid \mu_k, \Sigma_k).
$$

**Why "Gaussian mixture."** This density is a convex combination (weights $\phi_k\ge0$, $\sum_k\phi_k=1$) of $K$ individual Gaussian densities. Rather than a single unimodal Gaussian, $p(x_i)$ can display several modes/clusters — it "mixes" $K$ Gaussian populations together, hence *Gaussian mixture density*.

---

### Part 2: Deriving the Posterior Cluster Probability

By Bayes' rule applied to the discrete latent variable $C_i$:

$$
P(C_i=k \mid X_i=x_i) = \frac{P(X_i=x_i \mid C_i=k)\,P(C_i=k)}{\sum_{j=1}^{K} P(X_i=x_i \mid C_i=j)\,P(C_i=j)}.
$$

Substituting the Gaussian conditional density $P(X_i=x_i\mid C_i=k)=\mathcal N(x_i\mid\mu_k,\Sigma_k)$ and the prior $P(C_i=k)=\phi_k$:

$$
P(C_i=k \mid X_i=x_i) = \frac{\phi_k\,\mathcal N(x_i\mid\mu_k,\Sigma_k)}{\sum_{j=1}^{K}\phi_j\,\mathcal N(x_i\mid\mu_j,\Sigma_j)} \;=:\; \gamma_{ik}.
$$

**Why $\gamma_{ik}$ is a posterior probability.** $\phi_k$ is the *prior* belief about cluster membership before seeing any data. Once $x_i$ is observed, Bayes' rule reweights this prior by how compatible $x_i$ is with each cluster's Gaussian density, producing an updated (posterior) belief $\gamma_{ik}$ about which cluster generated $x_i$ — exactly the standard Bayesian update from prior to posterior given evidence.

---

### Part 3: One-Hot Encoding of the Latent Cluster Variable

Since $Z_{ik}\in\{0,1\}$ is a Bernoulli-type indicator with $Z_{ik}=1 \iff C_i=k$:

$$
\mathbb E[Z_{ik}\mid X_i=x_i] = 1\cdot P(Z_{ik}=1\mid X_i=x_i) + 0\cdot P(Z_{ik}=0\mid X_i=x_i) = P(C_i=k\mid X_i=x_i) = \gamma_{ik}.
$$

Stacking over $k=1,\dots,K$:

$$
\mathbb E[Z_i \mid X_i=x_i] = \begin{bmatrix}\gamma_{i1}\\ \gamma_{i2}\\ \vdots \\ \gamma_{iK}\end{bmatrix}.
$$

**Conclusion.** The soft cluster assignment of $x_i$ is precisely the conditional expectation of the (unobserved) one-hot cluster-membership vector given the observed data, $\mathbb E[Z_i\mid X_i=x_i]$ — a vector of posterior probabilities rather than a single hard label.

---

### Part 4: From Soft Assignment to Hard Clustering

A **hard** assignment collapses the full posterior vector down to a single deterministic label,

$$
\hat C_i = \arg\max_{1\le k\le K} \gamma_{ik},
$$

keeping only the most probable cluster and discarding all information about how confident or ambiguous that choice was. A **soft** assignment retains the entire vector $\mathbb E[Z_i\mid X_i=x_i]=(\gamma_{i1},\dots,\gamma_{iK})$, which expresses partial, probabilistic membership in every cluster simultaneously (e.g., a point near a cluster boundary might have $\gamma_{i1}=0.55,\ \gamma_{i2}=0.45$, signaling genuine ambiguity that a hard label would hide).

---

### Part 5: Conditional Expectation of the Observation Given the Cluster

By definition of the Gaussian conditional model, $X_i \mid C_i=k \sim \mathcal N(\mu_k,\Sigma_k)$, and the mean of a multivariate Gaussian is its own defining location parameter:

$$
\mathbb E[X_i \mid C_i=k] = \mu_k.
$$

**Why $\mu_k$ is the cluster center.** $\mu_k$ is, by construction, the average location of all observations that were generated from cluster $k$ — the point around which cluster $k$'s probability mass is centered.

**Comparing the two conditional expectations.**
- $\mathbb E[Z_i\mid X_i=x_i]$ goes **data $\to$ cluster**: given an *observed point*, it returns the (soft) probability that the point belongs to each cluster — this is the soft cluster *membership* of that specific observation.
- $\mathbb E[X_i\mid C_i=k]$ goes **cluster $\to$ data**: given a *cluster label*, it returns the expected (mean) location of data generated by that cluster — this is the *center* of the cluster in feature space.

They are converses of each other: one infers latent structure from data, the other predicts data from latent structure.

---

### Part 6: The Complete-Data Likelihood

If the labels $z_i$ were observed, the complete-data likelihood factorizes over data points and clusters using the indicator $z_{ik}$ as an exponent selector:

$$
p(x_1,\dots,x_n,z_1,\dots,z_n) = \prod_{i=1}^{n}\prod_{k=1}^{K}\big[\phi_k\,\mathcal N(x_i\mid\mu_k,\Sigma_k)\big]^{z_{ik}}.
$$

Taking logarithms (log turns the product into a sum, and the exponent $z_{ik}$ becomes a multiplicative coefficient):

$$
\ell_c = \sum_{i=1}^{n}\sum_{k=1}^{K} z_{ik}\Big[\log\phi_k + \log\mathcal N(x_i\mid\mu_k,\Sigma_k)\Big].
$$

**Why this is easy to maximize.** Because $z_{ik}\in\{0,1\}$ acts as a hard selector, for each $i$ only the single true cluster contributes a nonzero term. The double sum over $k$ therefore decouples into $K$ independent, ordinary Gaussian (and categorical, for $\phi_k$) maximum-likelihood problems — one per cluster — each solvable in closed form exactly as in fully-supervised Gaussian parameter estimation. The difficulty in practice is only that $z_{ik}$ is unknown.

---

### Part 7: The EM Interpretation

Since $Z_i$ is not observed, the E-step replaces each unknown indicator $z_{ik}$ by its conditional expectation given the observed data and the *current* parameter estimates (Part 3's result):

$$
z_{ik} \;\rightsquigarrow\; \mathbb E[Z_{ik}\mid X_i=x_i] = \gamma_{ik}.
$$

Substituting this into $\ell_c$ gives the expected complete-data log-likelihood:

$$
Q = \sum_{i=1}^{n}\sum_{k=1}^{K}\gamma_{ik}\Big[\log\phi_k + \log\mathcal N(x_i\mid\mu_k,\Sigma_k)\Big].
$$

**Why this is a conditional update.** The E-step does not hard-assign each point to one cluster; instead it recomputes, for every point, its posterior (conditional) probability of belonging to each cluster *given the model's current parameter estimates*. As the parameters change from iteration to iteration, these conditional membership probabilities $\gamma_{ik}$ are recomputed accordingly — it is a running, self-consistent update of belief about cluster membership, conditioned on the best information available at that iteration.

---

### Part 8: Parameter Updates

Maximizing $Q$ with respect to $\mu_k,\Sigma_k$ (unconstrained) and $\phi_k$ (subject to $\sum_k\phi_k=1$, via a Lagrange multiplier) yields the standard M-step updates. Defining the **effective cluster size**

$$
N_k = \sum_{i=1}^{n}\gamma_{ik},
$$

the updates are

$$
\phi_k^{\text{new}} = \frac{N_k}{n}, \qquad
\mu_k^{\text{new}} = \frac{1}{N_k}\sum_{i=1}^{n}\gamma_{ik}\,x_i, \qquad
\Sigma_k^{\text{new}} = \frac{1}{N_k}\sum_{i=1}^{n}\gamma_{ik}\,(x_i-\mu_k^{\text{new}})(x_i-\mu_k^{\text{new}})^{T}.
$$

**Interpretation of $\gamma_{ik}$ as a fractional weight.** Compare $\mu_k^{\text{new}}$ to the ordinary sample mean of "points known to be in cluster $k$" — here every point $x_i$ contributes to *every* cluster's mean and covariance, but weighted by its responsibility $\gamma_{ik}\in[0,1]$. A point almost certainly in cluster $k$ ($\gamma_{ik}\approx1$) contributes nearly a full unit of weight; a point that is ambiguous or belongs elsewhere ($\gamma_{ik}\approx0$) contributes almost nothing. $N_k=\sum_i\gamma_{ik}$ is thus the *effective* (fractional) number of points softly assigned to cluster $k$, generalizing a simple count to a continuous, uncertainty-aware weight.

---

### Part 9: Interpretation

Gaussian mixture clustering can be understood as a repeated cycle of conditional updating rather than a one-shot deterministic partition. The mixture weight $\phi_k$ encodes the *prior* probability of cluster $k$ before any specific point is considered. The Gaussian density $\mathcal N(x_i\mid\mu_k,\Sigma_k)$ then measures how compatible a given observation $x_i$ is with cluster $k$'s current location and shape. Combining these via Bayes' rule produces the responsibility $\gamma_{ik}$, the *posterior* probability that cluster $k$ generated $x_i$ after actually observing it — and stacking these responsibilities across all $k$ gives the soft assignment vector $\mathbb E[Z_i\mid X_i=x_i]$, the full probabilistic membership profile of that point. The M-step then re-estimates $\phi_k,\mu_k,\Sigma_k$ by treating these same responsibilities as fractional membership weights over the dataset. Because the E-step (computing $\gamma_{ik}$) and M-step (re-estimating parameters from $\gamma_{ik}$) alternate and feed into each other, Gaussian mixture clustering is fundamentally **probabilistic clustering built on repeatedly recomputing conditional expectations of a latent cluster-membership variable**, rather than fixing hard cluster labels once and for all.


In [3]:
"""
Part 10 — Computational Simulation and Out-of-Sample Validation
GMMFinancialSegmenter: a 2D Gaussian Mixture Model segmenter with Plotly diagnostics.

Requires: pip install scikit-learn plotly pandas numpy

Dataset: https://www.kaggle.com/datasets/arjunbhasin2013/ccdata
Download "CC GENERAL.csv" from the link above (requires a (free) Kaggle account/API
token — this sandbox has no network access to kaggle.com) and either:
  (a) upload it to your Colab session and pass its path to GMMFinancialSegmenter, or
  (b) leave csv_path=None to fall back to a synthetic 3-cluster dataset that mimics
      PURCHASES / CREDIT_LIMIT behavior, so the class can be tested end-to-end
      without the real file.
"""

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.mixture import GaussianMixture


class GMMFinancialSegmenter:
    """
    Two-dimensional Gaussian Mixture Model segmenter for financial behavior
    features (e.g., PURCHASES vs CREDIT_LIMIT), with EM fitting, out-of-sample
    log-likelihood validation, and interactive Plotly diagnostics.
    """

    def __init__(self, n_components=3, feature_x="PURCHASES", feature_y="CREDIT_LIMIT",
                 test_size=0.2, random_state=42):
        self.n_components = n_components
        self.feature_x = feature_x
        self.feature_y = feature_y
        self.test_size = test_size
        self.random_state = random_state

        self.scaler = StandardScaler()
        self.gmm = GaussianMixture(
            n_components=self.n_components,
            covariance_type="full",
            random_state=self.random_state,
        )

        self.df_raw = None
        self.X_train_raw = None
        self.X_test_raw = None
        self.X_train = None
        self.X_test = None

    # -----------------------------------------------------------------
    # 1. Data loading, splitting, and scaling
    # -----------------------------------------------------------------
    def _synthetic_fallback(self, n=2000):
        """Synthetic PURCHASES / CREDIT_LIMIT-like data with 3 latent segments."""
        rng = np.random.default_rng(self.random_state)
        n_k = n // 3
        # low spenders, low limit
        c1 = rng.multivariate_normal([300, 1500], [[6000, 2000], [2000, 40000]], n_k)
        # mid spenders, mid limit
        c2 = rng.multivariate_normal([1800, 5000], [[90000, 15000], [15000, 250000]], n_k)
        # high spenders, high limit
        c3 = rng.multivariate_normal([6000, 12000], [[400000, 60000], [60000, 900000]],
                                      n - 2 * n_k)
        data = np.vstack([c1, c2, c3])
        data = np.clip(data, 0, None)  # financial amounts can't be negative
        return pd.DataFrame(data, columns=[self.feature_x, self.feature_y])

    def load_and_prepare(self, csv_path=None):
        """Load the dataset (or fall back to synthetic data), select the two
        features, drop missing values, split 80/20, and standardize."""
        if csv_path is not None:
            df = pd.read_csv(csv_path)
            df = df[[self.feature_x, self.feature_y]].dropna()
            print(f"Loaded real dataset from '{csv_path}' with {len(df)} rows.")
        else:
            df = self._synthetic_fallback()
            print("No csv_path provided — using synthetic fallback data "
                  f"({len(df)} rows) mimicking {self.feature_x}/{self.feature_y}.")

        self.df_raw = df

        train_raw, test_raw = train_test_split(
            df.values, test_size=self.test_size, random_state=self.random_state
        )
        self.X_train_raw, self.X_test_raw = train_raw, test_raw

        self.X_train = self.scaler.fit_transform(train_raw)
        self.X_test = self.scaler.transform(test_raw)

        print(f"Train size: {len(self.X_train)}  |  Test size: {len(self.X_test)}")
        return self

    # -----------------------------------------------------------------
    # 2. EM execution
    # -----------------------------------------------------------------
    def fit(self):
        """Fit the GMM via EM on the standardized training data."""
        self.gmm.fit(self.X_train)
        print(f"Converged: {self.gmm.converged_}  |  Iterations used: {self.gmm.n_iter_}")
        return self

    # -----------------------------------------------------------------
    # 3. Out-of-sample performance
    # -----------------------------------------------------------------
    def evaluate(self):
        """Average per-sample log-likelihood of the model on the held-out test set."""
        avg_ll = self.gmm.score(self.X_test)  # sklearn already averages per sample
        print(f"Average out-of-sample log-likelihood (test set): {avg_ll:.4f}")
        return avg_ll

    # -----------------------------------------------------------------
    # Helper: build a fine coordinate grid in standardized feature space
    # and evaluate max posterior responsibility across it
    # -----------------------------------------------------------------
    def _responsibility_grid(self, n_grid=150, pad=0.1):
        x_min, x_max = self.X_train[:, 0].min(), self.X_train[:, 0].max()
        y_min, y_max = self.X_train[:, 1].min(), self.X_train[:, 1].max()
        x_range = x_max - x_min
        y_range = y_max - y_min

        xx, yy = np.meshgrid(
            np.linspace(x_min - pad * x_range, x_max + pad * x_range, n_grid),
            np.linspace(y_min - pad * y_range, y_max + pad * y_range, n_grid),
        )
        grid_std = np.column_stack([xx.ravel(), yy.ravel()])

        # posterior responsibilities gamma_ik across the grid (E-step formula)
        gamma_grid = self.gmm.predict_proba(grid_std)
        max_gamma = gamma_grid.max(axis=1).reshape(xx.shape)

        # convert grid back to raw units for plotting alongside raw data
        grid_raw = self.scaler.inverse_transform(grid_std)
        xx_raw = grid_raw[:, 0].reshape(xx.shape)
        yy_raw = grid_raw[:, 1].reshape(xx.shape)

        return xx_raw, yy_raw, max_gamma

    # -----------------------------------------------------------------
    # 4a. Empirical 2D density heatmap with marginals
    # -----------------------------------------------------------------
    def plot_density_heatmap(self):
        df_train = pd.DataFrame(self.X_train_raw, columns=[self.feature_x, self.feature_y])
        fig = px.density_heatmap(
            df_train, x=self.feature_x, y=self.feature_y,
            marginal_x="histogram", marginal_y="histogram",
            title="Empirical 2D Density Heatmap of Training Data (raw units)",
            template="plotly_white",
        )
        fig.update_layout(width=850, height=650)
        return fig

    # -----------------------------------------------------------------
    # 4b. Training assignment plot: contour of max responsibility + train pts
    # -----------------------------------------------------------------
    def plot_training_assignment(self):
        xx_raw, yy_raw, max_gamma = self._responsibility_grid()
        labels_train = self.gmm.predict(self.X_train)

        fig = go.Figure()
        fig.add_trace(go.Contour(
            x=xx_raw[0], y=yy_raw[:, 0], z=max_gamma,
            colorscale="Viridis", opacity=0.75,
            contours=dict(showlabels=True),
            colorbar=dict(title="max γ_ik"),
            name="max responsibility"
        ))
        fig.add_trace(go.Scatter(
            x=self.X_train_raw[:, 0], y=self.X_train_raw[:, 1],
            mode="markers",
            marker=dict(size=5, color=labels_train, colorscale="Turbo",
                        line=dict(width=0.5, color="white")),
            name="training points"
        ))
        fig.update_layout(
            title="Training Assignment: GMM Contours of Max Posterior Responsibility",
            xaxis_title=self.feature_x, yaxis_title=self.feature_y,
            template="plotly_white", width=850, height=650
        )
        return fig

    # -----------------------------------------------------------------
    # 4c. Test assignment plot: same contour + held-out test points
    # -----------------------------------------------------------------
    def plot_test_assignment(self):
        xx_raw, yy_raw, max_gamma = self._responsibility_grid()
        labels_test = self.gmm.predict(self.X_test)

        fig = go.Figure()
        fig.add_trace(go.Contour(
            x=xx_raw[0], y=yy_raw[:, 0], z=max_gamma,
            colorscale="Viridis", opacity=0.75,
            contours=dict(showlabels=True),
            colorbar=dict(title="max γ_ik"),
            name="max responsibility"
        ))
        fig.add_trace(go.Scatter(
            x=self.X_test_raw[:, 0], y=self.X_test_raw[:, 1],
            mode="markers",
            marker=dict(size=6, color=labels_test, colorscale="Turbo",
                        symbol="diamond", line=dict(width=0.5, color="white")),
            name="test points (out-of-sample)"
        ))
        fig.update_layout(
            title="Test Assignment: Out-of-Sample Points over Responsibility Contours",
            xaxis_title=self.feature_x, yaxis_title=self.feature_y,
            template="plotly_white", width=850, height=650
        )
        return fig

    # -----------------------------------------------------------------
    # Convenience: run the full pipeline end to end
    # -----------------------------------------------------------------
    def run_all(self, csv_path=None):
        self.load_and_prepare(csv_path=csv_path)
        self.fit()
        self.evaluate()
        fig_density = self.plot_density_heatmap()
        fig_train = self.plot_training_assignment()
        fig_test = self.plot_test_assignment()
        return fig_density, fig_train, fig_test


if __name__ == "__main__":
    # Set csv_path="CC GENERAL.csv" (or your uploaded path) to use the real
    # Kaggle dataset; leave as None to run on the synthetic fallback.
    segmenter = GMMFinancialSegmenter(n_components=3,
                                       feature_x="PURCHASES",
                                       feature_y="CREDIT_LIMIT")
    fig_density, fig_train, fig_test = segmenter.run_all(csv_path=None)

    fig_density.show()
    fig_train.show()
    fig_test.show()

No csv_path provided — using synthetic fallback data (2000 rows) mimicking PURCHASES/CREDIT_LIMIT.
Train size: 1600  |  Test size: 400
Converged: True  |  Iterations used: 2
Average out-of-sample log-likelihood (test set): 0.6219
